# 04. RoBERTa Backbone Baseline: CLS Pooling

Fundamentals of Natural Language / NLP-I, Universitat Autonoma de Barcelona, academic year 2025-2026. Team 10: Phoebe Iglesias, David Redrejo, and Pau Rossell.

After EDA, preprocessing, survey grounding, and classical baselines, we now test the first deep learning baseline: a Spanish biomedical-clinical RoBERTa encoder with a CLS-pooling classification head.

## Why RoBERTa Here?

RoBERTa is a Transformer-based pretrained language model. In this project, the backbone is `PlanTL-GOB-ES/roberta-base-biomedical-clinical-es`, which is appropriate because our literals are Spanish clinical/biomedical text rather than general-domain English text.

This connects to the NLP and Neural Networks material on Transformers and self-attention: instead of manually designing sparse TF-IDF features, the model starts from contextual subword representations learned during pretraining.

## CLS Pooling

The model uses the first token representation as the sequence feature: `features = hidden_states[:, 0, :]`. For RoBERTa, this first token corresponds to the special beginning token, usually written as `<s>`. We then apply `Dropout(0.1)` and a linear layer from hidden size 768 to 36 ICD category classes.

This is a simple and standard baseline. It does not yet test mean pooling, class weighting, augmentation, or ensembling; those belong to later model versions.

## Training Setup

The full run used AdamW, learning rate `2e-5`, weight decay `0.01`, batch size `128`, maximum `50` epochs, and early stopping with patience `10`. The best checkpoint is selected by validation accuracy. The default `max_length=32` comes from the earlier tokenizer analysis, where the literals were short and truncation at 32 tokens was not a practical issue.

In [ ]:
from pathlib import Path
import json
import pandas as pd

metrics = json.loads(Path('../outputs/metrics/v04_roberta_cls_metrics.json').read_text())
history = pd.read_csv('../outputs/logs/v04_roberta_cls_history.csv')
comparison = pd.DataFrame([
    {'model': 'v00_majority_baseline', 'accuracy': 0.125182, 'macro_f1': 0.006181, 'weighted_f1': 0.027854},
    {'model': 'v01_tfidf_char_logreg', 'accuracy': 0.522628, 'macro_f1': 0.402554, 'weighted_f1': 0.494943},
    {'model': 'v02_tfidf_word_svm', 'accuracy': 0.520073, 'macro_f1': 0.474196, 'weighted_f1': 0.514018},
    {'model': 'v03_similarity_retrieval_baseline', 'accuracy': 0.497445, 'macro_f1': 0.462789, 'weighted_f1': 0.496120},
    {'model': 'v04_roberta_cls', 'accuracy': metrics['accuracy'], 'macro_f1': metrics['macro_f1'], 'weighted_f1': metrics['weighted_f1']},
])
comparison


## Result

| model | validation accuracy | macro F1 | weighted F1 |
|---|---:|---:|---:|
| v04_roberta_cls | 0.5693 | 0.4943 | 0.5543 |

The best checkpoint was epoch 10. Training continued until epoch 20 because early stopping waited for 10 epochs without a new validation-accuracy improvement.

The provided/public CLS reference accuracy of 0.565 is useful context, but we do not confuse it with our reproduced Kaggle public result. Our 0.5693 is the internal validation accuracy on the stratified 80/20 split.

![RoBERTa CLS training curves](../reports/figures/fig_11_roberta_cls_training_curves.png)


In [ ]:
history.tail()


## Interpretation

This is the first model that clearly uses a pretrained contextual representation. It improves validation accuracy over the strongest classical baseline, which suggests that the biomedical-clinical RoBERTa backbone adds useful information beyond sparse lexical features.

However, macro F1 remains close to the best classical baselines. That matters because the competition target has 36 categories and the EDA showed imbalance. Our next deep learning steps should therefore test pooling strategies, class weighting, and possibly augmentation or retrieval-informed analysis rather than only chasing overall accuracy.